# Chuva máxima diária anual por município — Xavier BR-DWGD

Este notebook calcula, para cada ano de **1961 a 2025** e cada município brasileiro, a maior precipitação diária (mm/dia) da grade Xavier BR-DWGD. A extração é feita no **centróide municipal** do asset já verificado, portanto o resultado representa o pixel da grade de 0,1° que contém o centróide — não o máximo espacial dentro de todo o polígono municipal.

A coleção diária oficial é `projects/ee-alexandrexavier/assets/BR-DWGD`; a variável de precipitação é `pr` (mm). Ao final, a tabela é salva diretamente como JSON na mesma pasta em que o notebook é executado.

In [8]:
# Execute uma vez no Colab. Depois reinicie o ambiente se o Colab solicitar.
!pip install -q earthengine-api geemap pandas

In [9]:
import ee

# Informe o projeto Cloud habilitado para Earth Engine.
GEE_PROJECT = 'fcoliveira'

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

print('Earth Engine inicializado com sucesso.')


Earth Engine inicializado com sucesso.


## Parâmetros

Altere somente estes valores se necessário. O campo de código contém o caractere BOM (`\ufeff`) no asset de centroides atual.

In [10]:
ANO_INICIAL = 1961
ANO_FINAL = 2025 # inclusivo

XAVIER_ASSET = 'projects/ee-alexandrexavier/assets/BR-DWGD'
CENTROIDES_ASSET = 'projects/fcoliveira/assets/centroide_br'
COL_CODIGO = '﻿codigo_ibge'

# A resolução da grade Xavier é 0,1° (aprox. 11 km).
ESCALA_METROS = 11_000
TILE_SCALE = 4
ARQUIVO_SAIDA = f'xavier_chuva_maxima_diaria_anual_municipios_{ANO_INICIAL}_{ANO_FINAL}.json'

assert ANO_INICIAL <= ANO_FINAL, 'ANO_INICIAL deve ser menor ou igual a ANO_FINAL'

In [11]:
# Permite executar esta célula mesmo após uma reinicialização do kernel.
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)

chuva_diaria = ee.ImageCollection(XAVIER_ASSET).select('pr')
municipios = ee.FeatureCollection(CENTROIDES_ASSET).map(
    lambda feicao: ee.Feature(feicao.geometry(), {
        'codigo_ibge': ee.String(feicao.get(COL_CODIGO)),
        'nome_municipio': feicao.get('nome_municipio'),
        'uf': feicao.get('uf_sigla'),
    })
)

print('Bandas disponíveis:', chuva_diaria.first().bandNames().getInfo())
print('Centroides carregados:', municipios.size().getInfo())
print('Período solicitado:', ANO_INICIAL, 'a', ANO_FINAL)

Bandas disponíveis: ['pr']
Centroides carregados: 5571
Período solicitado: 1961 a 2025


## Máximo diário de cada ano

Para cada ano, a coleção diária é reduzida com `max()` (`imagem_maxima_anual`) e o valor desse raster anual é amostrado em todos os centroides (`calcular_maximo_anual`, via `reduceRegions`). A extração efetiva (`getInfo()`) é feita ano a ano na célula seguinte, para não estourar o limite de elementos de uma única chamada síncrona do Earth Engine (~5000 feições).

**Importante:** a banda `pr` do asset Xavier BR-DWGD vem armazenada como inteiro (int16), não em mm diretamente. Cada imagem carrega as properties `BAND_pr_MULT` e `BAND_pr_ADD` (confirmadas constantes de 1961 a 2025: `MULT=0.00686666`, `ADD=225`); o valor real é `mm = bruto * MULT + ADD`. Além disso, `reduceRegions` com `ee.Reducer.first()` sozinho nomeia a saída como `'first'`, não com o nome da banda — por isso o reducer usa `.setOutputs(['chuva_max_diaria_mm'])`, garantindo que a propriedade de saída tenha exatamente esse nome.


In [12]:
def _pixel_bruto_para_mm(imagem):
    mult = ee.Number(imagem.get('BAND_pr_MULT'))
    add = ee.Number(imagem.get('BAND_pr_ADD'))
    return imagem.multiply(mult).add(add).copyProperties(imagem, imagem.propertyNames())


def imagem_maxima_anual(ano):
    inicio = ee.Date.fromYMD(ano, 1, 1)
    fim = inicio.advance(1, 'year')

    return (
        chuva_diaria.filterDate(inicio, fim)
        .map(_pixel_bruto_para_mm)
        .max()
        .rename('chuva_max_diaria_mm')
    )


def calcular_maximo_anual(ano):
    maximo_anual = imagem_maxima_anual(ano)

    return maximo_anual.reduceRegions(
        collection=municipios,
        reducer=ee.Reducer.first().setOutputs(['chuva_max_diaria_mm']),
        scale=ESCALA_METROS,
        tileScale=TILE_SCALE,
    ).map(lambda feicao: feicao.set({
        'ano': ano,
        'fonte': 'Xavier BR-DWGD',
        'unidade': 'mm/dia',
    }))


## Extrair ano a ano e salvar JSON na pasta do notebook

Para cada ano, a `FeatureCollection` é baixada com `getInfo()` (consulta síncrona direta — nenhum dado é enviado para tasks de export do Earth Engine, nem Drive nem Assets). Como o Brasil tem mais de 5.000 municípios, uma única consulta síncrona por ano estoura o limite do Earth Engine (`Collection query aborted after accumulating over 5000 elements`) — por isso cada ano é baixado em páginas de `TAMANHO_PAGINA` municípios (`FeatureCollection.toList(count, offset)`), bem abaixo desse limite. Cada página tem até `MAX_TENTATIVAS` tentativas (com espera entre elas) em caso de erro transitório do Earth Engine.

Alguns municípios (tipicamente litorâneos/de borda) têm o centróide sobre um pixel sem dado válido na grade Xavier e voltam com `chuva_max_diaria_mm` nulo na amostragem direta. Para esses casos, o notebook faz uma segunda passada: amostra um buffer ao redor do ponto (raios crescentes em `RAIOS_PREENCHIMENTO_M`) e usa a **média dos pixels válidos vizinhos** (`ee.Reducer.mean()`, que ignora pixels mascarados) como valor de preenchimento. O campo `metodo` no registro final indica se o valor veio direto do centróide (`centroide`) ou de uma média de vizinhança (`media_vizinhanca_XXkm`), para preservar a rastreabilidade do dado.

O arquivo JSON é regravado a cada ano processado, para não perder o progresso acumulado caso um ano posterior falhe. Ao final, o notebook reporta registros que ficaram nulos mesmo após o preenchimento por vizinhança e anos cuja contagem de municípios ficou diferente do esperado.


In [13]:
import json
import time
from pathlib import Path

CAMPOS = ['codigo_ibge', 'nome_municipio', 'uf', 'ano', 'chuva_max_diaria_mm', 'unidade', 'fonte', 'metodo']
TOTAL_MUNICIPIOS = municipios.size().getInfo()
TAMANHO_PAGINA = 2000  # bem abaixo do limite de ~5000 elementos por consulta síncrona do EE
MAX_TENTATIVAS = 3
ESPERA_ENTRE_TENTATIVAS_S = 30
RAIOS_PREENCHIMENTO_M = [20_000, 50_000, 100_000]  # tentativas progressivas de raio para preencher nulos com a média da vizinhança

caminho_saida = Path.cwd() / ARQUIVO_SAIDA


def _getInfo_com_retry(objeto_ee, descricao):
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        try:
            return objeto_ee.getInfo()
        except Exception as erro:
            print(f'{descricao}: falha na tentativa {tentativa}/{MAX_TENTATIVAS} ({erro})')
            if tentativa < MAX_TENTATIVAS:
                time.sleep(ESPERA_ENTRE_TENTATIVAS_S)
    return None


def preencher_nulos_por_vizinhanca(imagem_ano, registros_por_codigo, pendentes):
    preenchidos = 0
    for raio in RAIOS_PREENCHIMENTO_M:
        if not pendentes:
            break
        alvo = municipios.filter(ee.Filter.inList('codigo_ibge', list(pendentes)))
        alvo_buffer = alvo.map(lambda feicao: feicao.setGeometry(feicao.geometry().buffer(raio)))

        resultado = _getInfo_com_retry(
            imagem_ano.reduceRegions(
                collection=alvo_buffer,
                reducer=ee.Reducer.mean().setOutputs(['chuva_max_diaria_mm']),
                scale=ESCALA_METROS,
                tileScale=TILE_SCALE,
            ),
            f'Preenchimento por vizinhança (raio {raio // 1000} km)',
        )
        if resultado is None:
            continue

        ainda_pendentes = set()
        for feicao in resultado['features']:
            propriedades = feicao['properties']
            codigo = propriedades.get('codigo_ibge')
            valor = propriedades.get('chuva_max_diaria_mm')
            if valor is None:
                ainda_pendentes.add(codigo)
            else:
                registros_por_codigo[codigo]['chuva_max_diaria_mm'] = valor
                registros_por_codigo[codigo]['metodo'] = f'media_vizinhanca_{raio // 1000}km'
                preenchidos += 1
        pendentes = ainda_pendentes

    return preenchidos, pendentes


registros = []
anos_com_falha = []

for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    resultado_ano = calcular_maximo_anual(ano)

    feicoes_ano = []
    ano_ok = True
    for inicio_pagina in range(0, TOTAL_MUNICIPIOS, TAMANHO_PAGINA):
        pagina_ok = False
        for tentativa in range(1, MAX_TENTATIVAS + 1):
            try:
                pagina = resultado_ano.toList(TAMANHO_PAGINA, inicio_pagina).getInfo()
                feicoes_ano.extend(pagina)
                pagina_ok = True
                break
            except Exception as erro:
                print(f'Ano {ano}, página a partir de {inicio_pagina}: falha na tentativa {tentativa}/{MAX_TENTATIVAS} ({erro})')
                if tentativa < MAX_TENTATIVAS:
                    time.sleep(ESPERA_ENTRE_TENTATIVAS_S)
        if not pagina_ok:
            ano_ok = False

    if not ano_ok:
        anos_com_falha.append(ano)

    registros_do_ano = {}
    for feicao in feicoes_ano:
        propriedades = feicao['properties']
        registro = {campo: propriedades.get(campo) for campo in CAMPOS if campo != 'metodo'}
        registro['metodo'] = 'centroide'
        registros_do_ano[registro['codigo_ibge']] = registro

    pendentes = {codigo for codigo, r in registros_do_ano.items() if r['chuva_max_diaria_mm'] is None}
    preenchidos = 0
    if pendentes:
        imagem_ano = imagem_maxima_anual(ano)
        preenchidos, pendentes = preencher_nulos_por_vizinhanca(imagem_ano, registros_do_ano, pendentes)

    registros.extend(registros_do_ano.values())

    nulos_no_ano = len(pendentes)
    aviso_nulos = ''
    if preenchidos:
        aviso_nulos += f', {preenchidos} preenchido(s) por média da vizinhança'
    if nulos_no_ano:
        aviso_nulos += f', {nulos_no_ano} ainda nulo(s) mesmo após vizinhança'
    aviso_contagem = ''
    if feicoes_ano and len(feicoes_ano) != TOTAL_MUNICIPIOS:
        aviso_contagem = f' [ATENÇÃO: esperado {TOTAL_MUNICIPIOS}]'
    print(f'Ano {ano}: {len(feicoes_ano)} municípios extraídos (total acumulado: {len(registros)}){aviso_nulos}{aviso_contagem}')

    # Regrava o JSON a cada ano processado, para não perder o progresso já
    # feito se um ano posterior falhar todas as tentativas.
    with open(caminho_saida, 'w', encoding='utf-8') as arquivo:
        json.dump(registros, arquivo, ensure_ascii=False, indent=2)

print()
print(f'JSON salvo em: {caminho_saida.resolve()}')
print(f'Total de registros: {len(registros)}')

total_nulos = sum(1 for r in registros if r['chuva_max_diaria_mm'] is None)
if total_nulos:
    print(f'ATENÇÃO: {total_nulos} registro(s) com chuva_max_diaria_mm nulo mesmo após preenchimento por vizinhança.')

if anos_com_falha:
    print(f'ATENÇÃO: anos com pelo menos uma página que falhou após {MAX_TENTATIVAS} tentativas: {anos_com_falha}')


Ano 1961: 5571 municípios extraídos (total acumulado: 5571), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1962: 5571 municípios extraídos (total acumulado: 11142), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1963: 5571 municípios extraídos (total acumulado: 16713), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1964: 5571 municípios extraídos (total acumulado: 22284), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1965: 5571 municípios extraídos (total acumulado: 27855), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1966: 5571 municípios extraídos (total acumulado: 33426), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 1967: 5571 municípios extraídos (total acumulado: 38997), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 196

Ano 2009: 5571 municípios extraídos (total acumulado: 272979), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2010: 5571 municípios extraídos (total acumulado: 278550), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2011: 5571 municípios extraídos (total acumulado: 284121), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2012: 5571 municípios extraídos (total acumulado: 289692), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2013: 5571 municípios extraídos (total acumulado: 295263), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2014: 5571 municípios extraídos (total acumulado: 300834), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança
Ano 2015: 5571 municípios extraídos (total acumulado: 306405), 19 preenchido(s) por média da vizinhança, 2 ainda nulo(s) mesmo após vizinhança

## Mapa de conferência (opcional)

Exibe o raster do máximo diário de um ano escolhido e os centroides. O mapa usa Folium/Leaflet, sem depender de chave do Google Maps.

In [14]:
ANO_MAPA = 2010
maximo_mapa = imagem_maxima_anual(ANO_MAPA)

try:
    import geemap.foliumap as geemap
except Exception as erro:
    geemap = None
    print(f'geemap indisponível para visualização (mapa opcional ignorado): {erro}')

if geemap is not None:
    Map = geemap.Map(location=[-15, -52], zoom_start=4)
    Map.add_layer(
        maximo_mapa,
        {'min': 0, 'max': 150, 'palette': ['ffffff', '9ecae1', '3182bd', '08519c', '67000d']},
        f'Máxima diária — {ANO_MAPA}',
    )
    Map.add_layer(municipios.style(**{'color': '222222', 'pointSize': 1}), {}, 'Centroides municipais')
    Map.add_layer_control()
    Map


geemap indisponível para visualização (mapa opcional ignorado): "'Box' object has no attribute 'xyz_to_folium'"


In [ ]:
from google.colab import files
files.download('/content/xavier_chuva_maxima_diaria_anual_municipios_1961_2025.json')
